# Demo: Microlensing MCP Server

This notebook shows how to run `hep_multiagent.Agent` with a running MCP server endpoint.

The microlensing server should be started separately and exposed as an MCP HTTP endpoint before running this notebook.

Prerequisites:

1. Install this package in your environment: `pip install -e .[dev]`
2. Start the MCP server separately.
3. Set `ARGO_BASE_URL`, `ARGO_API_KEY`, and optionally `ARGO_MODEL` and `MICROLENSING_MCP_SERVER_URL`.

The notebook uses ARGO as the LLM backend through LangChain's OpenAI-compatible `ChatOpenAI` client. To use another compatible backend, replace the `ChatOpenAI` configuration cell.


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from hep_multiagent import Agent

load_dotenv()

# These demos use ARGO's OpenAI-compatible chat API.
# Other OpenAI-compatible LLM providers can be used by changing these values.
llm = ChatOpenAI(
    model=os.environ.get("ARGO_MODEL", "gpt-5.5"),
    base_url=os.environ["ARGO_BASE_URL"],
    api_key=os.environ["ARGO_API_KEY"],
)

mcp_url = os.environ.get("MICROLENSING_MCP_SERVER_URL", "http://localhost:8002/mcp")
mcp_servers = [{"name": "mcp-microlensing", "url": mcp_url}]


In [ ]:
agent = await Agent(
    llm=llm,
    mcp_servers=mcp_servers,
)

print(f"Loaded {len(agent.tools)} MCP tools")
for tool in agent.tools[:20]:
    print(f"- {tool.name}")

In [ ]:
query = 'Use the microlensing MCP tools to evaluate a representative lensing scenario and summarize the resulting observables.'

result = await agent.run(
    query=query,
    output_dir="./output_demo_microlensing",
)

In [ ]:
print(result.get("final_report", result))

Generated files are written under the notebook-specific `output_*` directory. The most useful files are `execution_log.md`, `execution.ipynb`, and the final report artifacts.